# Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

# Path to your ZIP file
zip_path = '/content/drive/MyDrive/flair_images.zip'

# Directory where you want to extract the contents
extract_to = '/content/dataset'

# Create the directory if it doesn't exist
os.makedirs(extract_to, exist_ok=True)

# Extract the ZIP file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("ZIP file extracted successfully.")


In [ ]:
!pip install nibabel
!pip install torchvision


# Load Pre-trained Model

In [ ]:
import nibabel as nib
import numpy as np
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import csv
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
resnet = models.resnet50(pretrained=True)
resnet.eval()
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])


# Pre-processing of Images

In [ ]:
# Preprocess for ResNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def normalize_slice(slice_2d):
    s = (slice_2d - slice_2d.min()) / (slice_2d.max() - slice_2d.min() + 1e-8)
    s = (s * 255).astype(np.uint8)
    return s


# Extraction of Deep features

In [ ]:
import os
all_features = []

def extract_features(input_dir):
  global all_features # Declare all_features as global

  for fname in os.listdir(input_dir):
      if fname.endswith('.nii') :
          img = nib.load(os.path.join(input_dir, fname))
          volume = img.get_fdata()
          # repeat same extraction as above, save features per file


          for i in range(volume.shape[2]):  # each slice
              slice_norm = normalize_slice(volume[:, :, i])
              img_rgb = Image.fromarray(slice_norm).convert('RGB')
              img_tensor = transform(img_rgb).unsqueeze(0)  # (1,3,224,224)

              with torch.no_grad():
                  feats = feature_extractor(img_tensor)  # (1,2048,1,1)
                  feats = feats.view(feats.size(0), -1)  # (1,2048)

              all_features.append(feats.numpy().flatten())

extract_features('/content/dataset/flair_images_HGG')
extract_features('/content/dataset/flair_images_LGG')

all_features = np.array(all_features)  # Convert to numpy array after the loop
print('Feature array shape:', all_features.shape)

# save feature set

In [ ]:
df = pd.DataFrame(all_features)
df.to_csv('flair_features.csv', index=False, header=False)
print("Features saved to flair_features.csv")


In [ ]:
from google.colab import files
files.download('flair_features.csv')


In [ ]:
df.head()